# ChurnGuard EDA

This notebook profiles churn behavior using the same feature engineering layer used by the training pipeline.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.preprocess import RAW_DATA_PATH, engineer_features

sns.set_theme(style='whitegrid', palette='Set2')
FIGURE_DIR = Path('notebooks') / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)

In [ ]:
raw = pd.read_csv(RAW_DATA_PATH)
df = engineer_features(raw)
df['ChurnFlag'] = df['Churn'].map({'No': 0, 'Yes': 1})
df.head()

## Churn Rate Cuts

The first pass looks at churn by overall population, contract type, tenure segment, and payment method because those dimensions map directly to pricing, onboarding, and retention interventions.

In [ ]:
overall_churn = df['ChurnFlag'].mean()
contract_churn = df.groupby('Contract', observed=False)['ChurnFlag'].mean().sort_values(ascending=False)
tenure_churn = df.groupby('tenure_segment', observed=False)['ChurnFlag'].mean().sort_values(ascending=False)
payment_churn = df.groupby('PaymentMethod', observed=False)['ChurnFlag'].mean().sort_values(ascending=False)

display(pd.DataFrame({'metric': ['overall_churn_rate'], 'value': [overall_churn]}))
display(contract_churn.to_frame('churn_rate'))
display(tenure_churn.to_frame('churn_rate'))
display(payment_churn.to_frame('churn_rate'))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Overall churn'], [overall_churn], color='#d95f02')
ax.set_ylim(0, 1)
ax.set_ylabel('Churn rate')
ax.set_title('Overall Churn Rate')
ax.bar_label(ax.containers[0], labels=[f'{overall_churn:.1%}'], padding=4)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'churn_rate_overall.png', dpi=160)
plt.show()

In [ ]:
for column, filename, title in [
    ('Contract', 'churn_rate_by_contract.png', 'Churn Rate by Contract Type'),
    ('tenure_segment', 'churn_rate_by_tenure_segment.png', 'Churn Rate by Tenure Segment'),
    ('PaymentMethod', 'churn_rate_by_payment_method.png', 'Churn Rate by Payment Method'),
]:
    rates = df.groupby(column, observed=False)['ChurnFlag'].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.barplot(x=rates.index.astype(str), y=rates.values, ax=ax, color='#1b9e77')
    ax.set_ylim(0, max(0.6, rates.max() + 0.08))
    ax.set_xlabel(column)
    ax.set_ylabel('Churn rate')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=25)
    ax.bar_label(ax.containers[0], labels=[f'{v:.1%}' for v in rates.values], padding=3)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=160)
    plt.show()

## Engineered Feature Relationships

This heatmap focuses on engineered and core numeric features so correlation is readable and tied to model design choices.

In [ ]:
corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'service_bundle_count', 'monthly_to_total_ratio', 'contract_risk_score', 'ChurnFlag']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax)
ax.set_title('Correlation Heatmap: Engineered Features')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'engineered_feature_correlation_heatmap.png', dpi=160)
plt.show()

## Charge Distributions by Churn

Box plots show whether churned customers sit in different charge bands, which matters for retention economics.

In [ ]:
for column, filename, title in [
    ('MonthlyCharges', 'monthly_charges_vs_churn_boxplot.png', 'Monthly Charges vs Churn'),
    ('TotalCharges', 'total_charges_vs_churn_boxplot.png', 'Total Charges vs Churn'),
]:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.boxplot(data=df, x='Churn', y=column, ax=ax, hue='Churn', legend=False, palette=['#4c78a8', '#f58518'])
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=160)
    plt.show()